[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/59_batchnorm_full_solution.ipynb)

# Solution: BatchNorm Full Module

Reference solution.


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn


In [ ]:
# ✅ SOLUTION

class MyBatchNorm1d(nn.Module):
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        super().__init__()
        self.eps = eps
        self.momentum = momentum
        self.weight = nn.Parameter(torch.ones(num_features))
        self.bias = nn.Parameter(torch.zeros(num_features))
        self.register_buffer('running_mean', torch.zeros(num_features))
        self.register_buffer('running_var', torch.ones(num_features))

    def forward(self, x):
        if self.training:
            mean = x.mean(dim=0)
            var = x.var(dim=0, unbiased=False)
            self.running_mean.mul_(1 - self.momentum).add_(self.momentum * mean.detach())
            self.running_var.mul_(1 - self.momentum).add_(self.momentum * var.detach())
        else:
            mean = self.running_mean
            var = self.running_var
        x_hat = (x - mean) * torch.rsqrt(var + self.eps)
        return x_hat * self.weight + self.bias


In [ ]:
# Demo
m = MyBatchNorm1d(6)
x = torch.randn(8, 6)
print('Output:', m(x).shape)


In [ ]:
from torch_judge import check
check('batchnorm_full')
